In [ ]:
import sys; sys.path.append('..'); sys.path.append('../..')
import inflation, numpy as np, importlib, fd_validation, visualization, parametric_pillows, wall_generation
from numpy.linalg import norm
import MeshFEM, parallelism, benchmark, utils
import periodic_unit_helper
import numpy.linalg as la

In [ ]:
from periodic_simulation_setup import *

In [ ]:
h = 5

shift_range = np.arange(0, 2.5 - 0.75 - 0.5, 0.05)

In [ ]:
def analytic_scale(w, h):
    return (2 * (h - w) / np.pi + w) / h

In [ ]:
h = 2
w = 0.5
res = 1

triArea = h * w / res
avg_len = 0.05

In [ ]:
# h = 3
# w = 3

In [ ]:
shift = np.array([1.2, 0])

In [ ]:
import igl

In [ ]:
ipu, points, segment_edges, m, markers= periodic_unit_helper.get_shifted_dashline(h, w, avg_len, shift, angle = 37 / 180 * np.pi, two_dash = True, opposite_angle = True)

In [ ]:
visualization.plot_line_segments(points, segment_edges)

In [ ]:
visualization.plot_2d_mesh(m, pointList=np.where(markers)[0], width=5, height=5)

In [ ]:
finalMarkers = np.where(np.array(markers) == 1)[0]

In [ ]:
m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers)
# m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers)
# m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, axis = 1)
m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, axis = 1)



In [ ]:
visualization.plot_2d_mesh(m, pointList=finalMarkers, width=5, height=5)

In [ ]:
fusedVtx = get_fusedVtx_using_markers(len(m.vertices()), finalMarkers)

In [ ]:
import periodic_unit_helper

In [ ]:
fixedVars = periodic_unit_helper.get_center_fixedVars(ipu)

In [ ]:
# isheet.setRelaxedStiffnessEpsilon(1e-6)

In [ ]:
ipu = inflation.InflatablePeriodicUnit(m, fusedVtx = fusedVtx, epsilon = 1e-9)

In [ ]:
from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(ipu, width=768, height=640)
viewer.showWireframe(True)
viewer.show()

In [ ]:
viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])


In [ ]:
curr_vars = ipu.getVars()
curr_vars[-2] = 0.
ipu.setVars(curr_vars)
viewer.update()

In [ ]:
# Choose strategy for constraining rigid motion
fixedVars, hessianShift = periodic_unit_helper.get_center_fixedVars(ipu), 0
fixedVars, hessianShift = [ipu.numVars() - 2], 1e-6
# fixedVars, hessianShift = [], 1e-6

In [ ]:
ipu.sheet.setUseTensionFieldEnergy(True)
ipu.sheet.setUseHessianProjectedEnergy(False)
ipu.sheet.disableFusedRegionTensionFieldTheory(False)

ipu.sheet.pressure = 2

In [ ]:
benchmark.reset()

opts.niter = 1000
framerate = 5 # Update every 5 iterations
def cb(it):
    if it % framerate == 0:
        viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
cr = inflation.inflation_newton(ipu, fixedVars, opts, callback=cb, hessianShift = hessianShift)
benchmark.report()

In [ ]:
cr.success

In [ ]:
import importlib


In [ ]:
import periodic_simulation_setup

In [ ]:
importlib.reload(periodic_simulation_setup)

In [ ]:
import periodic_simulation_setup

In [ ]:
print("before", periodic_simulation_setup.allEnergies(ipu), periodic_simulation_setup.allGradientNorms(ipu))
periodic_simulation_setup.reparametrize_gamma_bar(ipu)
print("after ", periodic_simulation_setup.allEnergies(ipu), periodic_simulation_setup.allGradientNorms(ipu))



In [ ]:
visualize_sampled_bending_stiffness(ipu, 100, filename = "stiffness_shifted_dashline_{}.png".format(shift))

### Experiment 2

In [ ]:
analytic_sfs = []
simulated_sfs = []
second_sfs = []
counter = 0
for shift in shift_range:

    import igl

    shift = [5 / 6, shift]
    
    print(counter, shift)
    ipu, _, _, _, _ = periodic_unit_helper.get_shifted_dashline(h, w, avg_len, shift)

    fixedVars = periodic_unit_helper.get_center_fixedVars(ipu)

    # isheet.setRelaxedStiffnessEpsilon(1e-6)

    from tri_mesh_viewer import TriMeshViewer
    viewer = TriMeshViewer(ipu, width=768, height=640)
    viewer.showWireframe(True)
   
    ipu.sheet.rigidMotionPinVars

    fd_perturb = np.random.uniform(-1e-3, 1e-3, ipu.numVars())

    # ipu.setVars(ipu.getVars() + fd_perturb)

    import time, vis
    ipu.sheet.setUseTensionFieldEnergy(True)
    ipu.sheet.setUseHessianProjectedEnergy(False)
    ipu.sheet.pressure = 0.5
    opts.niter = 300
    framerate = 5 # Update every 5 iterations
    def cb(it):
        if it % framerate == 0:
            viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
    cr = inflation.inflation_newton(ipu, fixedVars, opts, callback=cb)
    
    render = viewer.offscreenRenderer(1000, 1000)
    render.render()
    render.save("shifted_dashline_inflated_with_width_{}.png".format(counter))
    counter += 1
    sfs = periodic_unit_helper.get_deformation_scale_factors(ipu)
    print("Simulated scale factor: ", sfs)
    simulated_sfs.append(sfs)
    print(periodic_unit_helper.get_second_deformation_scale_factors(ipu))
    second_sfs.append(periodic_unit_helper.get_second_deformation_scale_factors(ipu))

In [ ]:
np.save(

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
simulated_sfs

In [ ]:
fig, ax = plt.subplots()
# plt.plot(h_range, analytic_sfs, '-o', label="Analytic scale factors")
# plt.axhline(y = 0.7092958, linestyle = '-', color = 'orange', label = "Original scale factor")


plt.plot(shift_range, simulated_sfs, '-o', label="Simulated scale factors")
# plt.plot([5], [0.7092958], '-o', label="Original scale factor")


ax.legend()
plt.xlabel("wall height")
plt.ylabel("scale factor")
plt.savefig('shifted_dashline_scale_factor_comparison.png', dpi=300)

In [ ]:
analytic_scale(1, 5)

In [ ]:
fig, ax = plt.subplots()
# plt.plot(h_range, analytic_sfs, '-o', label="Analytic scale factors")
# plt.axhline(y = 0.7092958, linestyle = '-', color = 'orange', label = "Original scale factor")


plt.plot(simulated_sfs, second_sfs, '-o', label="Simulated scale factors")
# plt.plot([5], [0.7092958], '-o', label="Original scale factor")

plt.plot([min(simulated_sfs), max(second_sfs)] , [min(simulated_sfs), max(second_sfs)], '-')
# plt.plot([5], [0.7092958], '-o', label="Original scale factor")



ax.legend()
plt.xlabel("min scale factor")
plt.ylabel("max scale factor")
plt.savefig('shifted_dashline_two_scale_factor_comparison.png', dpi=300)

In [ ]:
second_sfs

In [ ]:
second_sfs